# Scaling analysis

Fits and diagnostics over the sweep in `runs/` (unzip `circscale_runs.zip`
there, or point `RUNS` elsewhere). Every post-warmup eval checkpoint of every
run is an (N, D = batch x step, L) datapoint (constant-LR schedule, so
mid-run points are honest); we fit the Chinchilla form
$L(N, D) = E + A N^{-\alpha} + B D^{-\beta}$ and decompose everything by
output tap depth.

Caveats to keep in mind: one seed per shape (seed noise ~0.002 in final loss,
measured from the w32d4 replicates below); the four smallest shapes tuned to
the LR grid edge (1e-2), so their losses may be slightly pessimistic.

In [ ]:
import glob
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import least_squares

from train import load_run

RUNS = "runs"
WARMUP_CUT = 1000   # drop warmup-phase checkpoints from fits
CHANCE = np.log(2)


def n_params(c):
    w, d, r = c["width"], c["mlp_depth"], c["hidden_ratio"]
    return c["n_wires"] * 2 * w + d * (2 * r * w * w + w) + w


runs = [load_run(p) for p in sorted(glob.glob(f"{RUNS}/*.npz"))]
grid = sorted([(c, d) for c, d in runs if c["model_seed"] == 0],
              key=lambda r: n_params(r[0]))
cmap = plt.cm.viridis(np.linspace(0, 0.95, len(grid)))

print(f"{'shape':10s} {'params':>11s} {'lr':>7s} {'final L':>8s} {'acc':>7s}")
for c, d in grid:
    print(f"w{c['width']}d{c['mlp_depth']:<4} {n_params(c):>11,d} {c['lr']:>7g} "
          f"{d['per_out_loss'][-1].mean():>8.4f} {d['per_out_acc'][-1].mean():>7.4f}")

noise = [d["per_out_loss"][-1].mean() for c, d in runs
         if (c["width"], c["mlp_depth"]) == (32, 4)]
print(f"\nseed noise (w32d4, n={len(noise)}): std {np.std(noise):.4f}")

## L(D): one curve per model size

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for (c, d), col in zip(grid, cmap):
    sel = d["eval_steps"] >= WARMUP_CUT
    ax.plot(d["eval_steps"][sel] * c["batch"], d["per_out_loss"][sel].mean(axis=1),
            color=col, lw=1.4, label=f"w{c['width']}d{c['mlp_depth']} ({n_params(c)/1e6:.2f}M)")
ax.axhline(CHANCE, color="gray", ls=":", lw=1)
ax.set(xscale="log", yscale="log", xlabel="samples D", ylabel="eval BCE (nats/bit)")
ax.legend(fontsize=7, ncol=2); ax.set_title("L(D) per model size (constant LR)")
plt.tight_layout()

## L(N) at the full data budget

In [ ]:
N = np.array([n_params(c) for c, _ in grid], dtype=float)
Lf = np.array([d["per_out_loss"][-1].mean() for _, d in grid])


def resid_sat(p):
    E, logA, alpha = p
    return np.log(E + np.exp(logA) * N ** (-alpha)) - np.log(Lf)

sat = min((least_squares(resid_sat, [0.55, 0.0, a0],
                         bounds=([0, -20, 0.01], [CHANCE, 20, 3]))
           for a0 in [0.1, 0.3, 0.5, 1.0]), key=lambda r: r.cost)
E1, A1, al1 = sat.x[0], np.exp(sat.x[1]), sat.x[2]
print(f"L(N) = {E1:.4f} + {A1:.3g} N^-{al1:.3f}  "
      f"(rms log resid {np.sqrt(2 * sat.cost / len(N)):.4f})")

fig, ax = plt.subplots(figsize=(6, 4.2))
ax.plot(N, Lf - E1, "o")
Ng = np.geomspace(N.min(), N.max(), 100)
ax.plot(Ng, A1 * Ng ** (-al1), "-",
        label=rf"$\alpha$={al1:.2f}, E={E1:.4f}")
ax.set(xscale="log", yscale="log", xlabel="params N", ylabel="L - E")
ax.legend(); ax.set_title("saturating power law, D=12.8M")
plt.tight_layout()

## Joint fit $L(N, D) = E + A N^{-\alpha} + B D^{-\beta}$

Huber loss on log residuals, multi-start.

In [ ]:
pN, pD, pL = [], [], []
for c, d in grid:
    sel = d["eval_steps"] >= WARMUP_CUT
    pN.append(np.full(sel.sum(), n_params(c)))
    pD.append(d["eval_steps"][sel] * c["batch"])
    pL.append(d["per_out_loss"][sel].mean(axis=1))
pN, pD, pL = map(np.concatenate, (pN, pD, pL))


def resid_joint(p):
    E, logA, alpha, logB, beta = p
    pred = E + np.exp(logA) * pN ** (-alpha) + np.exp(logB) * pD ** (-beta)
    return np.log(pred) - np.log(pL)

jf = min((least_squares(resid_joint, [0.5, 0.0, a0, 0.0, b0],
                        bounds=([0, -30, 0.01, -30, 0.01], [CHANCE, 30, 3, 30, 3]),
                        loss="huber", f_scale=0.02)
          for a0 in [0.2, 0.5] for b0 in [0.2, 0.5]), key=lambda r: r.cost)
E, A, al, B, be = jf.x[0], np.exp(jf.x[1]), jf.x[2], np.exp(jf.x[3]), jf.x[4]
pred = E + A * pN ** (-al) + B * pD ** (-be)
rms = np.sqrt(np.mean((np.log(pred) - np.log(pL)) ** 2))
print(f"L(N,D) = {E:.4f} + {A:.3g} N^-{al:.3f} + {B:.3g} D^-{be:.3f}  "
      f"(rms log resid {rms:.4f}, {len(pL)} points)")
print(f"compute-optimal: N_opt ~ C^{be/(al+be):.2f}, D_opt ~ C^{al/(al+be):.2f}")

fig, ax = plt.subplots(figsize=(4.6, 4.2))
ax.plot(pL, pred, ".", ms=3, alpha=0.4)
lims = [pL.min() * 0.98, pL.max() * 1.02]
ax.plot(lims, lims, "k-", lw=0.8)
ax.set(xlabel="observed L", ylabel="predicted L")
plt.tight_layout()

## Hardness ladder and solved-depth frontier

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
frontier = []
for (c, d), col in zip(grid, cmap):
    depths, acc = d["out_depths"], d["per_out_acc"][-1]
    uniq = np.arange(1, depths.max() + 1)
    macc = np.array([acc[depths == u].mean() if (depths == u).any() else np.nan
                     for u in uniq])
    axes[0].plot(uniq, macc, color=col, lw=1.3, label=f"w{c['width']}d{c['mlp_depth']}")
    solved = uniq[np.nan_to_num(macc) >= 0.9]
    frontier.append((n_params(c), solved.max() if len(solved) else 0))
axes[0].axhline(0.5, color="gray", ls=":", lw=1)
axes[0].set(xlabel="output tap depth", ylabel="final eval accuracy")
axes[0].legend(fontsize=7); axes[0].set_title("hardness ladder at D=12.8M")
fN, fD = zip(*frontier)
axes[1].semilogx(fN, fD, "o-")
axes[1].set(xlabel="params N", ylabel="max tap depth, mean acc >= 0.9",
            title="solved-depth frontier")
plt.tight_layout()

## Per-depth learning curves (largest model)

In [ ]:
c, d = grid[-1]
fig, ax = plt.subplots(figsize=(7, 4.5))
sel = d["eval_steps"] >= WARMUP_CUT
for k, col in zip(range(1, 9), plt.cm.plasma(np.linspace(0, 0.9, 8))):
    m = d["out_depths"] == k
    if m.any():
        ax.plot(d["eval_steps"][sel] * c["batch"],
                d["per_out_loss"][sel][:, m].mean(axis=1),
                color=col, lw=1.3, label=f"tap depth {k}")
ax.axhline(CHANCE, color="gray", ls=":", lw=1)
ax.set(xscale="log", yscale="log", xlabel="samples D", ylabel="eval BCE")
ax.legend(fontsize=7); ax.set_title(f"per-depth curves, w{c['width']}d{c['mlp_depth']}")
plt.tight_layout()

## Diagnosis: outputs stuck at chance are pure sparse parities

Shallow outputs split bimodally: almost all reach accuracy ~1.0, but a few sit
at exactly 0.5 even for the largest model. Checking their Fourier structure
against the true circuit shows each stuck output is a *pure parity* (XOR) of
its light-cone inputs — zero correlation with every proper subset, so there is
no low-degree signal for SGD to climb; learning a k-sparse parity over 256
inputs is the classic needle-in-a-haystack (Barak et al. 2022). The
"hardness ladder" is therefore really a Fourier-degree ladder: tap depth
raises hardness by pushing each output's spectrum toward higher-degree,
wider-support parities.

In [ ]:
from random_circuit import evaluate_np, sample_circuit

c0 = grid[0][0]
circuit = sample_circuit(np.random.default_rng(c0["circuit_seed"]),
                         c0["n_wires"], c0["circ_depth"])
c, d = grid[-1]
depths, acc = d["out_depths"], d["per_out_acc"][-1]
rng = np.random.default_rng(7)
x = rng.integers(0, 2, size=(8192, 256), dtype=np.uint8)
t = 1 - 2.0 * evaluate_np(circuit, x)
s = 1 - 2.0 * x

for w in np.flatnonzero((depths <= 3) & (acc < 0.6)):
    base = x[:256]
    y0 = evaluate_np(circuit, base)[:, w]
    deps = [i for i in range(256)
            if (evaluate_np(circuit, base ^ (np.arange(256) == i))[:, w] != y0).any()]
    corrs = {sub: np.mean(t[:, w] * np.prod(s[:, list(sub)], axis=1))
             for r in range(1, len(deps) + 1) for sub in combinations(deps, r)}
    top = sorted(corrs.items(), key=lambda kv: -abs(kv[1]))[:3]
    print(f"wire {w} (tap depth {depths[w]}, acc {acc[w]:.3f}): cone {deps}")
    for sub, corr in top:
        print(f"    corr with XOR{list(sub)}: {corr:+.3f}")